# 02. 토지형·건물형 공용 Test 예측·랭킹·SHAP

학습 노트북에서 저장한 Model Bundle을 불러와 후보지에 적용합니다.

이 노트북 코드는 토지형과 건물형에 공용으로 사용할 수 있습니다.

- Bundle의 `dataset_type`과 `feature_columns`를 자동으로 읽음
- CSV와 Excel Test 파일 모두 지원
- 모델 재학습 없이 `predict_proba()`로 점수와 순위 산출
- 후보지별 SHAP 추천요인 생성
- 상위 20개 JSON 생성

주의: 코드가 공용인 것이며 Test 데이터 파일은 토지형과 건물형을 각각 따로 사용해야 합니다.


In [14]:
!pip -q install xgboost lightgbm catboost shap joblib openpyxl

In [15]:
import shutil
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

print("라이브러리 로드 완료")


라이브러리 로드 완료


## 1. 모델과 Test 파일 업로드 및 설정

In [ ]:
UPLOAD_FILES = False

if UPLOAD_FILES:
    from google.colab import files
    uploaded = files.upload()
    print("업로드 파일:", list(uploaded.keys()))

# ============================================================
# 실행 대상 선택
# ============================================================
DATASET_TYPE = "land"
# DATASET_TYPE = "building"

# 입력 단계 선택
# "raw"    : 원본 후보 CSV
# "rule"   : Rule-based 검토를 통과한 CSV
# "vision" : Vision AI 결과가 결합된 CSV
INPUT_STAGE = "rule"

FILE_CONFIG = {
    "land": {
        "model": "Land_model_bundle.pkl",
        "raw": "Land_Test_Chungcheong_Uninstalled(4).csv",
        "rule": "Land_Test_Chungcheong_RulePassed.csv",
        "vision": "Land_Test_Chungcheong_VisionCompleted.csv",
    },
    "building": {
        "model": "Building_model_bundle.pkl",
        "raw": "Building_Test_Chungcheong_Uninstalled(2).csv",
        "rule": "Building_Test_Chungcheong_RulePassed.csv",
        "vision": "Building_Test_Chungcheong_VisionCompleted.csv",
    },
}

if DATASET_TYPE not in FILE_CONFIG:
    raise ValueError(
        "DATASET_TYPE은 'land' 또는 'building'이어야 합니다."
    )

if INPUT_STAGE not in {
    "raw",
    "rule",
    "vision",
}:
    raise ValueError(
        "INPUT_STAGE는 raw, rule, vision 중 하나여야 합니다."
    )

MODEL_BUNDLE_FILENAME = (
    FILE_CONFIG[DATASET_TYPE]["model"]
)

TEST_FILENAME = (
    FILE_CONFIG[DATASET_TYPE][INPUT_STAGE]
)

MODEL_BUNDLE_PATH = (
    Path("/content")
    / MODEL_BUNDLE_FILENAME
)

TEST_PATH = (
    Path("/content")
    / TEST_FILENAME
)

# Rule 결과 파일을 직접 사용하는 경우에도 추가 안전 필터
FILTER_RULE_EXCLUDED = True

# "label_0": label=0만 랭킹
# "all": 입력 후보 전체 랭킹
RANK_FILTER_MODE = "all"

CREATE_REGION_RANKS = True
SHAP_TOP_N = 1000

# 기본값은 순수 ML 점수
ML_WEIGHT = 1.0
POLICY_WEIGHT = 0.0

POLICY_WEIGHT_CONFIG = {}

if not MODEL_BUNDLE_PATH.exists():
    raise FileNotFoundError(
        f"모델 번들을 찾을 수 없습니다: {MODEL_BUNDLE_PATH}"
    )

if not TEST_PATH.exists():
    raise FileNotFoundError(
        f"Test 파일을 찾을 수 없습니다: {TEST_PATH}"
    )

print("DATASET_TYPE:", DATASET_TYPE)
print("INPUT_STAGE:", INPUT_STAGE)
print("MODEL:", MODEL_BUNDLE_FILENAME)
print("TEST:", TEST_FILENAME)


모델 Bundle: /content/Land_model_bundle.pkl
Test 파일: /content/Land_Test_Chungcheong_Uninstalled.csv
랭킹 대상: all


## 2. Model Bundle 및 Test 데이터 로드

In [17]:
bundle = joblib.load(MODEL_BUNDLE_PATH)

model = bundle["model"]
model_name = bundle["model_name"]
dataset_type = str(
    bundle.get("dataset_type", "land")
).lower()

feature_columns = bundle["feature_columns"]
target_column = bundle.get("target_column", "label")
metadata_columns = bundle.get(
    "metadata_columns",
    [],
)
feature_korean_names = bundle.get(
    "feature_korean_names",
    {},
)
train_medians = pd.Series(
    bundle["train_medians"]
)

if dataset_type not in ["land", "building"]:
    raise ValueError(
        "dataset_type은 land 또는 building이어야 합니다."
    )

suffix = TEST_PATH.suffix.lower()

if suffix in [".xlsx", ".xls"]:
    test_df = pd.read_excel(TEST_PATH)
elif suffix == ".csv":
    test_df = pd.read_csv(TEST_PATH)
else:
    raise ValueError(
        "Test 파일은 xlsx, xls 또는 csv 형식이어야 합니다."
    )


# Rule-base 결과 컬럼이 있으면 확정 EXCLUDE만 제거
# LEGAL_REVIEW, UNKNOWN, NO_APPLICABLE_RULE은 그대로 랭킹 후보에 유지
if (
    FILTER_RULE_EXCLUDED
    and "Rule_Pass_For_Next_Step" in test_df.columns
):
    before_rule_filter = len(test_df)

    rule_pass_mask = (
        test_df["Rule_Pass_For_Next_Step"]
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes", "y"])
    )

    test_df = (
        test_df[rule_pass_mask]
        .copy()
        .reset_index(drop=True)
    )

    print(
        "Rule 확정 제외 제거:",
        before_rule_filter - len(test_df),
        "건",
    )

OUTPUT_PREFIX = (
    "Land"
    if dataset_type == "land"
    else "Building"
)

OUTPUT_DIR = Path(
    f"/content/{dataset_type}_test_ranking_results"
)
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("대상 유형:", dataset_type.upper())
print("모델:", model_name)
print("Feature 수:", len(feature_columns))
print("Test 크기:", test_df.shape)
print("결과 폴더:", OUTPUT_DIR)

missing_features = [
    col for col in feature_columns
    if col not in test_df.columns
]

if missing_features:
    raise KeyError(
        "Test 데이터에 모델 Feature가 없습니다: "
        f"{missing_features}"
    )

for col in feature_columns:
    test_df[col] = pd.to_numeric(
        test_df[col],
        errors="coerce",
    )

test_df = test_df.reset_index(drop=True).copy()
test_df["test_row_id"] = np.arange(len(test_df))

X_test = (
    test_df[feature_columns]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(train_medians)
)

remaining_missing = (
    X_test.columns[
        X_test.isna().any()
    ].tolist()
)

if remaining_missing:
    raise ValueError(
        "Train 중앙값 적용 후에도 Test 결측치가 남았습니다: "
        f"{remaining_missing}"
    )

print("예측 입력 크기:", X_test.shape)


대상 유형: LAND
모델: LightGBM
Feature 수: 23
Test 크기: (532, 34)
결과 폴더: /content/land_test_ranking_results
예측 입력 크기: (532, 23)


## 3. 전체 Test 예측 및 성능평가

In [18]:
# 랭킹에는 분류 결과가 아니라 양성 클래스 확률만 사용
test_probability = model.predict_proba(X_test)[:, 1]

scored_test = test_df.copy()

scored_test[
    "Solar_Readiness_Probability"
] = test_probability

scored_test["ML_Score"] = (
    scored_test[
        "Solar_Readiness_Probability"
    ]
    * 100
).round(4)

preview_columns = [
    col for col in [
        "source_id_ml",
        "address_ml",
        "시도",
        "시군구",
        "Solar_Readiness_Probability",
        "ML_Score",
    ]
    if col in scored_test.columns
]

display(
    scored_test[preview_columns]
    .sort_values(
        "ML_Score",
        ascending=False,
    )
    .head(20)
)


,source_id_ml,address_ml,시도,시군구,Solar_Readiness_Probability,ML_Score
216,SOLAR_01484,충청남도 부여군 규암면 규암리 104-24,충청남도,부여군,1.0000,99.9954
467,SOLAR_01773,음성군 삼성면 덕정리 1052-36,충청북도,NaN,0.9989,99.8884
507,SOLAR_01649,진천군 이월면 노원리 1003-34,충청북도,NaN,0.9988,99.8847
36,SOLAR_01304,충청남도 홍성군 홍동면 금당리 133-1,충청남도,홍성군,0.9985,99.8498
253,SOLAR_01380,충청남도 부여군 구룡면 태양리 146-3,충청남도,부여군,0.9984,99.8369
102,SOLAR_01421,충청남도 서천군 화양면 월산리 271-1,충청남도,서천군,0.9980,99.7992
376,SOLAR_01293,충청남도 서천군 화양면 월산리 263-1,충청남도,서천군,0.9977,99.7672
213,SOLAR_01762,청주시 흥덕구 옥산면 국사리 601-1,충청북도,흥덕구,0.9976,99.7640
404,SOLAR_01629,진천군 이월면 노원리 185-85,충청북도,NaN,0.9974,99.7408
35,SOLAR_01408,충청남도 금산군 군북면 두두리 375-6,충청남도,금산군,0.9971,99.7062


## 4. 정책 Feature 점수 함수

In [19]:
def calculate_policy_score(
    data,
    weight_config,
):
    if not weight_config:
        return pd.Series(
            0.0,
            index=data.index,
            dtype=float,
        )

    total_weight = sum(
        config["weight"]
        for config in weight_config.values()
    )

    if total_weight <= 0:
        raise ValueError(
            "정책 Feature 가중치 합은 0보다 커야 합니다."
        )

    policy_score = pd.Series(
        0.0,
        index=data.index,
        dtype=float,
    )

    for column, config in weight_config.items():
        if column not in data.columns:
            raise KeyError(
                f"정책 가중치 컬럼이 없습니다: {column}"
            )

        values = pd.to_numeric(
            data[column],
            errors="coerce",
        )

        fallback = train_medians.get(
            column,
            values.median(),
        )

        values = values.fillna(fallback)

        normalized_score = values.rank(
            pct=True,
            method="average",
        )

        direction = config["direction"]

        if direction == "lower":
            normalized_score = 1 - normalized_score
        elif direction != "higher":
            raise ValueError(
                f"{column} direction은 "
                "'higher' 또는 'lower'여야 합니다."
            )

        normalized_weight = (
            config["weight"] / total_weight
        )

        data[f"{column}_Policy_Score"] = (
            normalized_score
        )

        policy_score += (
            normalized_score
            * normalized_weight
        )

    return policy_score


## 5. Test 후보지 점수·순위·등급 산출

In [20]:
if RANK_FILTER_MODE == "label_0":
    if target_column not in scored_test.columns:
        raise KeyError(
            "label_0 모드는 Test 데이터에 label이 필요합니다."
        )

    candidate_ranking = (
        scored_test[
            pd.to_numeric(
                scored_test[target_column],
                errors="coerce",
            ) == 0
        ]
        .copy()
    )

elif RANK_FILTER_MODE == "all":
    candidate_ranking = scored_test.copy()

else:
    raise ValueError(
        "RANK_FILTER_MODE는 'label_0' 또는 'all'이어야 합니다."
    )

if candidate_ranking.empty:
    raise ValueError("랭킹 대상 후보지가 없습니다.")

candidate_ranking["Policy_Feature_Score"] = (
    calculate_policy_score(
        candidate_ranking,
        POLICY_WEIGHT_CONFIG,
    )
)

candidate_ranking["Final_Readiness_Probability"] = (
    ML_WEIGHT
    * candidate_ranking["Solar_Readiness_Probability"]
    +
    POLICY_WEIGHT
    * candidate_ranking["Policy_Feature_Score"]
)

candidate_ranking["Solar_Readiness_Score"] = (
    candidate_ranking["Final_Readiness_Probability"]
    * 100
).round(4)

candidate_ranking["Candidate_Rank"] = (
    candidate_ranking["Solar_Readiness_Score"]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)

candidate_ranking["Score_Percentile"] = (
    candidate_ranking["Solar_Readiness_Score"]
    .rank(
        pct=True,
        ascending=True,
    )
)

candidate_ranking["Solar_Readiness_Grade"] = pd.cut(
    candidate_ranking["Score_Percentile"],
    bins=[0, 0.50, 0.80, 1.00],
    labels=["C", "B", "A"],
    include_lowest=True,
)

if CREATE_REGION_RANKS:
    if "시도" in candidate_ranking.columns:
        candidate_ranking["Province_Rank"] = (
            candidate_ranking
            .groupby(
                "시도",
                dropna=False,
            )["Solar_Readiness_Score"]
            .rank(
                method="min",
                ascending=False,
            )
            .astype(int)
        )

    if (
        "시도" in candidate_ranking.columns
        and "시군구" in candidate_ranking.columns
    ):
        candidate_ranking["Local_Rank"] = (
            candidate_ranking
            .groupby(
                ["시도", "시군구"],
                dropna=False,
            )["Solar_Readiness_Score"]
            .rank(
                method="min",
                ascending=False,
            )
            .astype(int)
        )

candidate_ranking = (
    candidate_ranking
    .sort_values(
        [
            "Solar_Readiness_Score",
            "Candidate_Rank",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

ranking_view_columns = [
    col for col in metadata_columns
    if col in candidate_ranking.columns
] + [
    "Solar_Readiness_Probability",
    "ML_Score",
    "Policy_Feature_Score",
    "Solar_Readiness_Score",
    "Candidate_Rank",
    "Score_Percentile",
    "Solar_Readiness_Grade",
]

for optional_col in [
    "Province_Rank",
    "Local_Rank",
]:
    if optional_col in candidate_ranking.columns:
        ranking_view_columns.append(optional_col)

display(
    candidate_ranking[
        ranking_view_columns
    ].head(30)
)


,source_id_ml,address_ml,longitude,latitude,시도,시군구,자산구분_ML,설치구분,region_group,Solar_Readiness_Probability,ML_Score,Policy_Feature_Score,Solar_Readiness_Score,Candidate_Rank,Score_Percentile,Solar_Readiness_Grade,Province_Rank,Local_Rank
0,SOLAR_01484,충청남도 부여군 규암면 규암리 104-24,126.8882,36.2738,충청남도,부여군,토지,미설치,충청남도,1.0000,99.9954,0.0000,99.9954,1,1.0000,A,1,1
1,SOLAR_01773,음성군 삼성면 덕정리 1052-36,127.4971,37.0152,충청북도,NaN,토지,미설치,충청북도,0.9989,99.8884,0.0000,99.8884,2,0.9981,A,1,1
2,SOLAR_01649,진천군 이월면 노원리 1003-34,127.4195,36.9146,충청북도,NaN,토지,미설치,충청북도,0.9988,99.8847,0.0000,99.8847,3,0.9962,A,2,2
3,SOLAR_01304,충청남도 홍성군 홍동면 금당리 133-1,126.7427,36.5579,충청남도,홍성군,토지,미설치,충청남도,0.9985,99.8498,0.0000,99.8498,4,0.9944,A,2,1
4,SOLAR_01380,충청남도 부여군 구룡면 태양리 146-3,126.8132,36.2578,충청남도,부여군,토지,미설치,충청남도,0.9984,99.8369,0.0000,99.8369,5,0.9925,A,3,2
5,SOLAR_01421,충청남도 서천군 화양면 월산리 271-1,126.8161,36.0637,충청남도,서천군,토지,미설치,충청남도,0.9980,99.7992,0.0000,99.7992,6,0.9906,A,4,1
6,SOLAR_01293,충청남도 서천군 화양면 월산리 263-1,126.8165,36.0632,충청남도,서천군,토지,미설치,충청남도,0.9977,99.7672,0.0000,99.7672,7,0.9887,A,5,2
7,SOLAR_01762,청주시 흥덕구 옥산면 국사리 601-1,127.3945,36.6863,충청북도,흥덕구,토지,미설치,충청북도,0.9976,99.7640,0.0000,99.7640,8,0.9868,A,3,1
8,SOLAR_01629,진천군 이월면 노원리 185-85,127.4325,36.9064,충청북도,NaN,토지,미설치,충청북도,0.9974,99.7408,0.0000,99.7408,9,0.9850,A,4,3
9,SOLAR_01408,충청남도 금산군 군북면 두두리 375-6,127.5273,36.1685,충청남도,금산군,토지,미설치,충청남도,0.9971,99.7062,0.0000,99.7062,10,0.9831,A,6,1


## 6. 후보지별 SHAP 추천요인

In [21]:
def extract_positive_class_values(shap_result):
    values = shap_result.values

    if isinstance(values, list):
        values = values[-1]

    values = np.asarray(values)

    if values.ndim == 3:
        values = values[:, :, 1]

    return values


def build_reason_sentence(
    feature_name,
    feature_value,
    percentile,
    shap_value,
):
    return (
        f"{feature_name} 값이 {feature_value:.3f}이며 "
        f"현재 Test 후보지 기준 {percentile:.1f}백분위입니다. "
        f"이 값은 모델의 설치 가능성 점수를 높인 주요 요인으로 "
        f"분석되었습니다 (SHAP {shap_value:+.4f})."
    )


top_n = min(
    SHAP_TOP_N,
    len(candidate_ranking),
)

top_candidates = (
    candidate_ranking
    .head(top_n)
    .copy()
)

top_test_row_ids = (
    top_candidates["test_row_id"]
    .astype(int)
    .tolist()
)

X_top = (
    scored_test
    .set_index("test_row_id")
    .loc[top_test_row_ids, feature_columns]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(train_medians)
)

explainer = shap.TreeExplainer(model)
shap_result = explainer(X_top)
shap_values = extract_positive_class_values(
    shap_result
)

reference_test_row_ids = (
    candidate_ranking["test_row_id"]
    .astype(int)
    .tolist()
)

X_reference = (
    scored_test
    .set_index("test_row_id")
    .loc[reference_test_row_ids, feature_columns]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(train_medians)
)

percentile_reference = (
    X_reference
    .rank(pct=True)
    * 100
)

reason_rows = []

for row_position in range(len(top_candidates)):
    candidate = top_candidates.iloc[row_position]
    test_row_id = int(candidate["test_row_id"])

    one_candidate = pd.DataFrame({
        "Feature": feature_columns,
        "Feature_Value": X_top.iloc[row_position].values,
        "SHAP_Value": shap_values[row_position],
    })

    one_candidate = (
        one_candidate[
            one_candidate["SHAP_Value"] > 0
        ]
        .assign(
            Abs_SHAP=lambda data: (
                data["SHAP_Value"].abs()
            )
        )
        .sort_values(
            "Abs_SHAP",
            ascending=False,
        )
        .head(3)
        .copy()
    )

    for reason_rank, (_, reason) in enumerate(
        one_candidate.iterrows(),
        start=1,
    ):
        feature = reason["Feature"]
        feature_name = feature_korean_names.get(
            feature,
            feature,
        )
        percentile = float(
            percentile_reference.loc[
                test_row_id,
                feature,
            ]
        )

        reason_rows.append({
            "test_row_id": test_row_id,
            "Candidate_Rank": int(
                candidate["Candidate_Rank"]
            ),
            "Reason_Rank": reason_rank,
            "Feature": feature,
            "Feature_Korean": feature_name,
            "Feature_Value": float(
                reason["Feature_Value"]
            ),
            "SHAP_Value": float(
                reason["SHAP_Value"]
            ),
            "Percentile": percentile,
            "Reason_Text": build_reason_sentence(
                feature_name=feature_name,
                feature_value=float(
                    reason["Feature_Value"]
                ),
                percentile=percentile,
                shap_value=float(
                    reason["SHAP_Value"]
                ),
            ),
        })

candidate_shap_details = pd.DataFrame(
    reason_rows
)

if not candidate_shap_details.empty:
    feature_wide = (
        candidate_shap_details
        .pivot(
            index="test_row_id",
            columns="Reason_Rank",
            values="Feature_Korean",
        )
        .rename(columns={
            1: "추천요인_1",
            2: "추천요인_2",
            3: "추천요인_3",
        })
        .reset_index()
    )

    reason_wide = (
        candidate_shap_details
        .pivot(
            index="test_row_id",
            columns="Reason_Rank",
            values="Reason_Text",
        )
        .rename(columns={
            1: "추천이유_1",
            2: "추천이유_2",
            3: "추천이유_3",
        })
        .reset_index()
    )

    candidate_ranking_with_shap = (
        candidate_ranking
        .merge(
            feature_wide,
            on="test_row_id",
            how="left",
        )
        .merge(
            reason_wide,
            on="test_row_id",
            how="left",
        )
    )
else:
    candidate_ranking_with_shap = (
        candidate_ranking.copy()
    )

for number in [1, 2, 3]:
    for prefix in ["추천요인", "추천이유"]:
        column = f"{prefix}_{number}"

        if (
            column
            not in candidate_ranking_with_shap.columns
        ):
            candidate_ranking_with_shap[column] = pd.NA

shap_view_columns = (
    ranking_view_columns
    + [
        "추천요인_1",
        "추천이유_1",
        "추천요인_2",
        "추천이유_2",
        "추천요인_3",
        "추천이유_3",
    ]
)

display(
    candidate_ranking_with_shap[
        shap_view_columns
    ].head(30)
)


,source_id_ml,address_ml,longitude,latitude,시도,시군구,자산구분_ML,설치구분,region_group,Solar_Readiness_Probability,ML_Score,Policy_Feature_Score,Solar_Readiness_Score,Candidate_Rank,Score_Percentile,Solar_Readiness_Grade,Province_Rank,Local_Rank,추천요인_1,추천이유_1,추천요인_2,추천이유_2,추천요인_3,추천이유_3
0,SOLAR_01484,충청남도 부여군 규암면 규암리 104-24,126.8882,36.2738,충청남도,부여군,토지,미설치,충청남도,1.0000,99.9954,0.0000,99.9954,1,1.0000,A,1,1,산란일사량,산란일사량 값이 2.038이며 현재 Test 후보지 기준 100.0백분위입니다. 이...,예상 태양광 발전량,예상 태양광 발전량 값이 3.843이며 현재 Test 후보지 기준 71.6백분위입니...,100m 높이 풍속,100m 높이 풍속 값이 4.197이며 현재 Test 후보지 기준 49.6백분위입니...
1,SOLAR_01773,음성군 삼성면 덕정리 1052-36,127.4971,37.0152,충청북도,NaN,토지,미설치,충청북도,0.9989,99.8884,0.0000,99.8884,2,0.9981,A,1,1,수평면 전일사량,수평면 전일사량 값이 4.029이며 현재 Test 후보지 기준 57.4백분위입니다....,평균 기온,평균 기온 값이 12.600이며 현재 Test 후보지 기준 32.0백분위입니다. 이...,예상 태양광 발전량,예상 태양광 발전량 값이 3.863이며 현재 Test 후보지 기준 84.1백분위입니...
2,SOLAR_01649,진천군 이월면 노원리 1003-34,127.4195,36.9146,충청북도,NaN,토지,미설치,충청북도,0.9988,99.8847,0.0000,99.8847,3,0.9962,A,2,2,5km 이내 전력선 길이,5km 이내 전력선 길이 값이 30.737이며 현재 Test 후보지 기준 93.6백...,예상 태양광 발전량,예상 태양광 발전량 값이 3.785이며 현재 Test 후보지 기준 27.9백분위입니...,10m 높이 풍속,10m 높이 풍속 값이 0.463이며 현재 Test 후보지 기준 9.2백분위입니다....
3,SOLAR_01304,충청남도 홍성군 홍동면 금당리 133-1,126.7427,36.5579,충청남도,홍성군,토지,미설치,충청남도,0.9985,99.8498,0.0000,99.8498,4,0.9944,A,2,1,10m 높이 풍속,10m 높이 풍속 값이 0.406이며 현재 Test 후보지 기준 5.6백분위입니다....,평균 기온,평균 기온 값이 12.600이며 현재 Test 후보지 기준 32.0백분위입니다. 이...,예상 태양광 발전량,예상 태양광 발전량 값이 3.796이며 현재 Test 후보지 기준 32.9백분위입니...
4,SOLAR_01380,충청남도 부여군 구룡면 태양리 146-3,126.8132,36.2578,충청남도,부여군,토지,미설치,충청남도,0.9984,99.8369,0.0000,99.8369,5,0.9925,A,3,2,산란일사량,산란일사량 값이 2.030이며 현재 Test 후보지 기준 97.7백분위입니다. 이 ...,100m 높이 풍속,100m 높이 풍속 값이 4.306이며 현재 Test 후보지 기준 54.9백분위입니...,예상 태양광 발전량,예상 태양광 발전량 값이 3.852이며 현재 Test 후보지 기준 79.2백분위입니...
5,SOLAR_01421,충청남도 서천군 화양면 월산리 271-1,126.8161,36.0637,충청남도,서천군,토지,미설치,충청남도,0.9980,99.7992,0.0000,99.7992,6,0.9906,A,4,1,산란일사량,산란일사량 값이 2.033이며 현재 Test 후보지 기준 98.2백분위입니다. 이 ...,예상 태양광 발전량,예상 태양광 발전량 값이 3.873이며 현재 Test 후보지 기준 89.1백분위입니...,평균 기온,평균 기온 값이 13.800이며 현재 Test 후보지 기준 99.7백분위입니다. 이...
6,SOLAR_01293,충청남도 서천군 화양면 월산리 263-1,126.8165,36.0632,충청남도,서천군,토지,미설치,충청남도,0.9977,99.7672,0.0000,99.7672,7,0.9887,A,5,2,산란일사량,산란일사량 값이 2.033이며 현재 Test 후보지 기준 98.2백분위입니다. 이 ...,예상 태양광 발전량,예상 태양광 발전량 값이 3.873이며 현재 Test 후보지 기준 89.1백분위입니...,평균 기온,평균 기온 값이 13.800이며 현재 Test 후보지 기준 99.7백분위입니다. 이...
7,SOLAR_01762,청주시 흥덕구 옥산면 국사리 601-1,127.3945,36.6863,충청북도,흥덕구,토지,미설치,충청북도,0.9976,99.7640,0.0000,99.7640,8,0.9868,A,3,1,5km 이내 전력선 길이,5km 이내 전력선 길이 값이 34.001이며 현재 Test 후보지 기준 95.5백...,최근접 전력선 거리,최근접 전력선 거리 값이 0.046이며 현재 Test 후보지 기준 0.4백분위입니다...,지형 음영도,지형 음영도 값이 121.778이며 현재 Test 후보지 기준 3.9백분위입니다. ...
8,SOLAR_01629,진천군 이월면 노원리 185-85,127.4325,36.9064,충청북도,NaN,토지,미설치,충청북도,0.9974,99.7408,0.0000,99.7408,9,0.9850,A,4,3,예상 태양광 발전량,예상 태양광 발전량 값이 3.832이며 현재 Test 후보지 기준 63.4백분위입니...,산란일사량,산란일사량 값이 2.011이며 현재 Test 후보지 기준 72.7백분위입니다. 이 ...,100m 높이 풍속,100m 높이 풍속 값이 3.872이며 현재 Test 후보지 기준 25.2백분위입니...
9,SOLAR_01408,충청남도 금산군 군북면 두두리 375-6,127.5273,36.1685,충청남도,금산군,토지,미설치,충청남도,0.9971,99.7062,0.0000,99.7062,10,0.9831,A,6,1,예상 태양광 발전량,예상 태양광 발전량 값이 3.813이며 현재 Test 후보지 기준 46.5백분위입니...,100m 높이 풍속,100m 높이 풍속 값이 3.497이며 현재 Test 후보지 기준 7.5백분위입니다...,인근 전력선 최대 전압,인근 전력선 최대 전압 값이 345.000이며 현재 Test 후보지 기준 93.2백...


In [22]:
import json
from datetime import datetime

# SHAP 결과가 이미 결합되어 있다면 그것을 사용하고,
# 아직 SHAP 실행 전이라면 candidate_ranking을 사용
if "candidate_ranking_with_shap" in globals():
    json_source_df = candidate_ranking_with_shap.copy()
else:
    json_source_df = candidate_ranking.copy()


# --------------------------------------------------
# 1. JSON 변환 보조 함수
# --------------------------------------------------

def is_valid_value(value):
    """None, NaN, pd.NA 여부 확인"""
    if value is None:
        return False

    try:
        return not pd.isna(value)
    except (TypeError, ValueError):
        return True


def safe_value(value, default=None):
    """pandas/numpy 값을 JSON 직렬화 가능한 값으로 변환"""
    if not is_valid_value(value):
        return default

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        return float(value)

    if isinstance(value, np.bool_):
        return bool(value)

    return value


def safe_float(value, digits=2, default=None):
    if not is_valid_value(value):
        return default

    try:
        return round(float(value), digits)
    except (TypeError, ValueError):
        return default


def safe_int(value, default=None):
    if not is_valid_value(value):
        return default

    try:
        return int(round(float(value)))
    except (TypeError, ValueError):
        return default


def first_available_value(
    row,
    candidate_columns,
    default=None,
):
    """
    여러 후보 컬럼 중 실제로 존재하고 값이 있는
    첫 번째 컬럼의 값을 반환
    """
    for column in candidate_columns:
        if (
            column in row.index
            and is_valid_value(row[column])
        ):
            return safe_value(row[column])

    return default


def degree_to_direction(value):
    """0~360도 방위각을 8방위 문자열로 변환"""
    degree = safe_float(value)

    if degree is None:
        return None

    directions = [
        "북향",
        "북동향",
        "동향",
        "남동향",
        "남향",
        "남서향",
        "서향",
        "북서향",
    ]

    index = int(
        ((degree + 22.5) % 360) // 45
    )

    return directions[index]


def make_status(grade):
    """
    프로젝트 내부 표시용 상태 기준.
    실제 인허가 통과를 의미하는 값은 아님.
    """
    grade = str(grade)

    if grade in ["A", "B"]:
        return "통과"

    if grade == "C":
        return "재검토"

    return "검토 필요"


def make_grid_description(row):
    """
    OSM 기반 계통 접근성 정보만 표시.
    실제 계통 여유 용량을 의미하지 않음.
    """
    substation_distance = safe_float(
        row.get("distance_to_substation_km")
    )

    powerline_distance = safe_float(
        row.get("distance_to_powerline_km")
    )

    descriptions = []

    if substation_distance is not None:
        descriptions.append(
            f"최근접 변전소 약 "
            f"{substation_distance:.2f}km"
        )

    if powerline_distance is not None:
        descriptions.append(
            f"최근접 전력선 약 "
            f"{powerline_distance:.2f}km"
        )

    if not descriptions:
        return None

    return (
        ", ".join(descriptions)
        + " 거리로 분석됨 "
        + "(공개 공간정보 기반 접근성)"
    )


def collect_text_values(
    row,
    columns,
):
    """추천이유 등 문자열 목록 생성"""
    result = []

    for column in columns:
        if (
            column in row.index
            and is_valid_value(row[column])
        ):
            text = str(row[column]).strip()

            if text:
                result.append(text)

    return result

In [23]:
# --------------------------------------------------
# 후보지 한 건을 공용 JSON 형식으로 변환
# --------------------------------------------------

def get_type_settings(dataset_type):
    if dataset_type == "building":
        return {
            "target_type": "BUILDING",
            "id_prefix": "BUILDING",
            "default_space_type": "건물형 후보지",
            "checklist": [
                {
                    "item": "지붕 구조안전성 및 적재하중 확인",
                    "note": "구조검토를 통해 태양광 모듈·구조물 하중 수용 가능 여부 확인",
                },
                {
                    "item": "옥상 방수 상태 및 보수 필요 여부",
                    "note": "누수 이력과 방수층 수명을 확인하고 필요 시 설치 전 보수",
                },
                {
                    "item": "옥상 장애물·설비 및 피난동선 확인",
                    "note": "냉난방기, 물탱크, 피뢰설비와 소방 피난동선 간섭 여부 확인",
                },
                {
                    "item": "계통연계 및 건축·소방 기준 검토",
                    "note": "한전 계통연계 가능 여부와 건축물·소방 관련 기준 확인",
                },
            ],
        }

    return {
        "target_type": "LAND",
        "id_prefix": "LAND",
        "default_space_type": "토지형 후보지",
        "checklist": [
            {
                "item": "진입로 확보 및 공사차량 진입 가능 여부",
                "note": "현장 진입로 폭과 도로점용 허가 필요 여부 확인",
            },
            {
                "item": "토질 상태 및 배수시설 설치 가능 여부",
                "note": "우수 배제와 토사 유출 가능성 현장 확인",
            },
            {
                "item": "최근접 전력선 및 계통연계 가능 여부",
                "note": "한전 계통 여유 용량과 선로 신설 비용 별도 확인",
            },
            {
                "item": "개발행위허가 및 이격거리 충족 여부",
                "note": "해당 지자체의 현행 조례와 부지 경계를 기준으로 재검토",
            },
        ],
    }


TYPE_SETTINGS = get_type_settings(dataset_type)


def build_candidate_json(row):
    ml_score = safe_float(
        row.get("ML_Score"),
        digits=2,
    )

    total_score = safe_float(
        row.get("Solar_Readiness_Score"),
        digits=2,
    )

    policy_score = None

    if POLICY_WEIGHT > 0 and POLICY_WEIGHT_CONFIG:
        policy_raw = safe_float(
            row.get("Policy_Feature_Score"),
            digits=4,
        )

        if policy_raw is not None:
            policy_score = round(
                policy_raw * 100,
                2,
            )

    priority_rank = first_available_value(
        row,
        [
            "Local_Rank",
            "Province_Rank",
            "Candidate_Rank",
        ],
    )

    if priority_rank is not None:
        priority_rank = str(
            safe_int(priority_rank)
        )

    bonus_reasons = collect_text_values(
        row,
        [
            "추천이유_1",
            "추천이유_2",
            "추천이유_3",
        ],
    )

    penalty_reasons = collect_text_values(
        row,
        [
            "감점이유_1",
            "감점이유_2",
            "감점이유_3",
        ],
    )

    if bonus_reasons:
        ml_reason = bonus_reasons[0]
    else:
        probability = safe_float(
            row.get(
                "Solar_Readiness_Probability"
            ),
            digits=4,
        )

        ml_reason = (
            f"저장된 ML 모델이 산출한 "
            f"설치사례 유사 확률은 "
            f"{probability * 100:.2f}%입니다."
            if probability is not None
            else None
        )

    rule_reason = first_available_value(
        row,
        [
            "Rule_Final_Message",
            "rule_final_message",
        ],
    )

    if rule_reason is None:
        rule_reason = (
            "사용자가 설정한 정책 Feature 가중치를 "
            "최종 점수에 반영함"
            if policy_score is not None
            else "Rule-based 검토 결과가 연결되지 않음"
        )

    site_id = first_available_value(
        row,
        [
            "source_id_ml",
            "후보ID",
            "site_id",
        ],
        default=(
            f"{TYPE_SETTINGS['id_prefix']}_"
            f"{safe_int(row.name, 0):05d}"
        ),
    )

    address = first_available_value(
        row,
        [
            "address_ml",
            "주소",
            "소재지",
            "도로명주소",
            "지번주소",
        ],
    )

    site_name = first_available_value(
        row,
        [
            "site_name",
            "재산명",
            "시설명",
            "자산명",
            "발전소명",
            "건물명",
        ],
    )

    if site_name is None:
        site_name = (
            f"{address} 태양광 후보지"
            if address
            else f"{site_id} 태양광 후보지"
        )

    space_type = first_available_value(
        row,
        [
            "space_type",
            "자산구분_ML",
            "설치구분",
            "재산구분",
            "지목",
            "건물용도",
        ],
        default=TYPE_SETTINGS[
            "default_space_type"
        ],
    )

    total_area = safe_float(
        first_available_value(
            row,
            [
                "total_area",
                "전체면적",
                "토지면적",
                "연면적",
                "옥상면적",
                "면적",
                "공유재산면적",
                "대지면적",
            ],
        )
    )

    available_area = safe_float(
        first_available_value(
            row,
            [
                "available_area",
                "설치가능면적",
                "가용면적",
                "유효면적",
                "옥상가용면적",
            ],
        )
    )

    availability_rate = None

    if (
        total_area is not None
        and available_area is not None
        and total_area > 0
    ):
        availability_rate = round(
            available_area
            / total_area
            * 100,
            2,
        )

    owner_agency = first_available_value(
        row,
        [
            "owner_agency",
            "소유기관",
            "관리기관",
            "기관명",
            "소관기관",
        ],
    )

    grade = safe_value(
        row.get("Solar_Readiness_Grade")
    )

    if dataset_type == "land":
        slope_degree = safe_float(
            row.get("slope_avg"),
            digits=2,
        )
        aspect_direction = degree_to_direction(
            row.get("slope_dir")
        )
    else:
        slope_degree = None
        aspect_direction = None

    return {
        "target_type": TYPE_SETTINGS[
            "target_type"
        ],

        "1_site_info": {
            "site_id": str(site_id),
            "site_name": site_name,
            "address": address,
            "space_type": space_type,
            "total_area": total_area,
            "available_area": available_area,
            "availability_rate_percent": (
                availability_rate
            ),
            "owner_agency": owner_agency,
            "created_at": datetime.now().strftime(
                "%Y년 %m월 %d일"
            ),
        },

        "2_scores_and_evaluation": {
            "grade": (
                str(grade)
                if grade is not None
                else None
            ),
            "total_score": total_score,
            "priority_rank": priority_rank,
            "status": make_status(grade),

            "detail_scores": {
                "ml_technical_score": ml_score,
                "ml_reason": ml_reason,
                "vision_ai_score": safe_float(
                    first_available_value(
                        row,
                        [
                            "Vision_Score",
                            "vision_score",
                        ],
                    ),
                    digits=2,
                ),
                "vision_reason": first_available_value(
                    row,
                    [
                        "Vision_Final_Message",
                        "vision_reason",
                    ],
                    default=(
                        "Vision AI 분석 결과가 "
                        "아직 연결되지 않음"
                    ),
                ),
                "rule_based_score": policy_score,
                "rule_reason": rule_reason,
            },

            "xai_explanation": {
                "bonus_reason": bonus_reasons,
                "penalty_reason": penalty_reasons,
            },
        },

        "3_vision_ai_simulation": {
            "vision_analysis": {
                "slope_degree": slope_degree,
                "aspect_direction": aspect_direction,
                "vegetation_coverage_percent": None,
                "has_access_road": None,
                "access_road_width_m": None,
                "recommended_orientation": None,
                "recommended_tilt_angle_deg": None,
            },

            "simulation": {
                "recommended_capacity_kw": None,
                "annual_generation_kwh": None,
                "annual_revenue_krw": None,
                "roi_percent": None,
                "payback_years": None,
            },
        },

        "4_risk_and_support": {
            "rule_based_risk_check": {
                "grid_connection": (
                    make_grid_description(row)
                ),
                "regulation": first_available_value(
                    row,
                    [
                        "Rule_Final_Message",
                        "rule_final_message",
                    ],
                ),
                "public_complaint": None,
            },
            "recommended_subsidies": [],
        },

        "5_pre_investigation_checklist": (
            TYPE_SETTINGS["checklist"]
        ),
    }


In [24]:
# --------------------------------------------------
# 3. 점수 상위 20개만 JSON 생성
# --------------------------------------------------

top20_df = (
    json_source_df
    .sort_values(
        [
            "Solar_Readiness_Score",
            "Candidate_Rank",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .head(20)
    .copy()
)

top20_candidate_json = [
    build_candidate_json(row)
    for _, row in top20_df.iterrows()
]

# 전체 20개 JSON 출력
print(
    json.dumps(
        top20_candidate_json,
        ensure_ascii=False,
        indent=2,
    )
)

[
  {
    "target_type": "LAND",
    "1_site_info": {
      "site_id": "SOLAR_01484",
      "site_name": "충청남도 부여군 규암면 규암리 104-24 태양광 후보지",
      "address": "충청남도 부여군 규암면 규암리 104-24",
      "space_type": "토지",
      "total_area": null,
      "available_area": null,
      "availability_rate_percent": null,
      "owner_agency": null,
      "created_at": "2026년 07월 27일"
    },
    "2_scores_and_evaluation": {
      "grade": "A",
      "total_score": 100.0,
      "priority_rank": "1",
      "status": "통과",
      "detail_scores": {
        "ml_technical_score": 100.0,
        "ml_reason": "산란일사량 값이 2.038이며 현재 Test 후보지 기준 100.0백분위입니다. 이 값은 모델의 설치 가능성 점수를 높인 주요 요인으로 분석되었습니다 (SHAP +5.7948).",
        "vision_ai_score": null,
        "vision_reason": "Vision AI 분석 결과가 아직 연결되지 않음",
        "rule_based_score": null,
        "rule_reason": "현재 랭킹에는 별도의 Rule-based 가중치를 적용하지 않음"
      },
      "xai_explanation": {
        "bonus_reason": [
          "산란일사량 값이 2.038이며 현재 Test 후보지 기준 100.0백분위입니다. 이 값

In [25]:
# --------------------------------------------------
# 상위 20개 JSON 파일 저장
# --------------------------------------------------

json_output_path = (
    OUTPUT_DIR
    / f"{OUTPUT_PREFIX}_Top20_Candidate_Analysis.json"
)

with open(
    json_output_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        top20_candidate_json,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("JSON 저장 완료:", json_output_path)


JSON 저장 완료: /content/land_test_ranking_results/Land_Top20_Candidate_Analysis.json


## 7. 결과 저장 및 다운로드

In [26]:
ranking_csv_path = (
    OUTPUT_DIR
    / f"{OUTPUT_PREFIX}_Test_candidate_ranking.csv"
)

ranking_shap_csv_path = (
    OUTPUT_DIR
    / f"{OUTPUT_PREFIX}_Test_candidate_ranking_with_shap.csv"
)

shap_detail_csv_path = (
    OUTPUT_DIR
    / f"{OUTPUT_PREFIX}_Test_candidate_shap_details.csv"
)

excel_output_path = (
    OUTPUT_DIR
    / f"{OUTPUT_PREFIX}_Test_ranking_results.xlsx"
)

candidate_ranking.to_csv(
    ranking_csv_path,
    index=False,
    encoding="utf-8-sig",
)

candidate_ranking_with_shap.to_csv(
    ranking_shap_csv_path,
    index=False,
    encoding="utf-8-sig",
)

candidate_shap_details.to_csv(
    shap_detail_csv_path,
    index=False,
    encoding="utf-8-sig",
)

with pd.ExcelWriter(
    excel_output_path,
    engine="openpyxl",
) as writer:
    scored_test.to_excel(
        writer,
        sheet_name="전체_Test_예측",
        index=False,
    )

    candidate_ranking_with_shap.to_excel(
        writer,
        sheet_name="후보지_랭킹_SHAP",
        index=False,
    )

    candidate_shap_details.to_excel(
        writer,
        sheet_name="SHAP_세부",
        index=False,
    )

zip_base = (
    f"/content/{dataset_type}_test_ranking_results"
)

zip_path = Path(
    shutil.make_archive(
        zip_base,
        "zip",
        OUTPUT_DIR,
    )
)

print("결과 저장 완료:", OUTPUT_DIR)
print("압축 완료:", zip_path)

from google.colab import files

files.download(str(excel_output_path))
files.download(str(zip_path))
files.download(str(json_output_path))


결과 저장 완료: /content/land_test_ranking_results
압축 완료: /content/land_test_ranking_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>